# NB19 — Simplified TFT Encoder

**Goal:** Train a Temporal Fusion Transformer (TFT) encoder on 15-lap sequence windows and entity embeddings, targeting Spearman ρ ~0.78–0.83 vs MLP NB15 to provide genuine diversity in the Phase 3 super-ensemble.

## Why TFT Instead of FT-Transformer?

The FT-Transformer (originally planned for this slot) processes each row **independently** using feature-level attention — `LapTime_lag1`, `LapTime_lag2`, `LapTime_lag3` are just three unordered feature tokens; temporal ordering is ignored entirely.

The TFT Encoder is designed for **mixed static + time-varying inputs** with explicit temporal ordering. Our data maps directly onto TFT's input taxonomy:

| TFT Input Type | Our Features |
|---|---|
| Static covariates | Driver, Race, Year, Compound + 38 engineered scalars |
| Time-varying observed past | LapTime, TyreLife, Cumulative_Degradation, LapTime_Delta, Position, PitStop |

## The Key Novelty: Variable Selection Networks (VSN)

VSNs learn **which features matter at each lap** via a softmax gating mechanism. No other model in the pipeline does this:
- At lap 5 (fresh tyres, early stint): `TyreLife` and degradation features get low weight
- At lap 38 (tyre cliff imminent, late race): `Cumulative_Degradation` and `laps_remaining` dominate

The MLP applies fixed weights to all features simultaneously. The LSTM routes all 6 sequence features through the same hidden state. Only VSN learns time-dependent feature relevance.

## Architecture

```
ENTITY EMBEDDINGS
  Driver(887→32) + Race(26→8) + Compound(5→3) + Year(4→2) = 45 dims
  emb_proj(45 → d_model=64) + ReLU  →  entity_ctx (B, 64)
         │
         ▼ [context]
STATIC VSN (38 scalar features)
  Per-feature Linear(1→64), stack → (B, 38, 64)
  GRN(38×64, ctx=entity_ctx) → softmax weights → weighted sum
  → static_vsn (B, 64)
         │
         ▼
STATIC MERGE  [cat[static_vsn, entity_ctx] → GRN(128→64)]
  → static_repr (B, 64)   ← conditions LSTM init + temporal VSN
         │
    ┌────┴──────────────────────────────────────────────────┐
    │ h0/c0 init                       temporal context     │
    ▼                                         ▼             │
h0 = Linear(64→2×64)           TEMPORAL VSN (6 seq features × 15 laps)
c0 = Linear(64→2×64)             VSN(6, d=64, ctx=static_repr)
    └────────┬──────────────────────────┘                   │
             ▼                                              │
     LSTM(64→64, 2 layers, batch_first=True)               │
       pack_padded_sequence (skip zero-padded steps)       │
       lstm_out (B, 15, 64)                                │
             │                                             │
             ▼                                             │
     MULTI-HEAD SELF-ATTENTION (4 heads)                  │
       key_padding_mask=~mask (ignore padding)            │
       attn_out = GRN(LayerNorm(lstm_out + attn))         │
       masked mean pool → pooled (B, 64)                  │
             │                                            │
             ▼                                            │
     GRN(64→64) → Linear(64→1) → logit                   │
```

**Versus NB18 LSTM:** static context initialises the LSTM (shapes dynamics from step 1) rather than concatenated at output; VSN dynamically routes which of the 6 sequence features matter per step; attention over all LSTM outputs captures long-range lap dependencies.

## Training Config
- `WINDOW=15` laps (shorter than NB18's 20 — attention compensates for shorter raw history)
- `D_MODEL=64`, `NUM_HEADS=4`, `NUM_LAYERS=2`, `DROPOUT=0.1`
- `BCEWithLogitsLoss(pos_weight=4.03)`, `AdamW(lr=5e-4, wd=1e-4)`, `CosineAnnealingLR(T_max=50)`
- `BATCH_SIZE=2048`, `MAX_EPOCHS=80`, `PATIENCE=10`
- Target runtime: ~30–45 min/fold on Kaggle T4 GPU

## Inputs / Outputs
**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`, `oof_predictions_nb15.parquet` (MLP), `oof_predictions_nb18.parquet` (LSTM), `oof_predictions_nb12.parquet` (LGBM)

**Outputs:** `oof_predictions_nb19.parquet` (`tft_oof`), `test_predictions_nb19.parquet` (`tft_pred`), `models/tft_fold{0-4}.pt`, `models/nb19_summary.pkl`

## Phase 3 Ensemble Gate
- Solo OOF AUC ≥ **0.905** (raised from 0.895; current best solo is LSTM at 0.9122)
- Spearman ρ < **0.90** vs MLP NB15
- Both gates must pass for inclusion in NB20 rank average

In [ ]:
# nb19-01  Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# nb19-02  Path detection + data load
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(exist_ok=True)
print(f'Root: {project_root}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
mlp_oof  = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')[['id', 'mlp_oof']]
lstm_oof = pd.read_parquet(processed_dir / 'oof_predictions_nb18.parquet')[['id', 'lstm_oof']]
lgbm_oof = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')[['id', 'lgbm_tuned_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

In [ ]:
# nb19-03  Settings
FEAT_COLS = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq', 'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session', 'Stint', 'Position',
    'Degradation_rate', 'Degradation_acceleration', 'Cumulative_Degradation_winsorized',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3', 'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3', 'PitStop_lag1',
    'laps_to_driver_end', 'field_median_laptime', 'laptime_vs_field', 'field_pace_change',
]
assert len(FEAT_COLS) == 38, f'Expected 38, got {len(FEAT_COLS)}'

SEQ_COLS = ['LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
            'LapTime_Delta', 'Position', 'PitStop']
CAT_COLS = ['Driver', 'Race', 'Compound', 'Year']

WINDOW     = 15        # shorter than NB18; attention handles longer-range
D_MODEL    = 64
NUM_HEADS  = 4
NUM_LAYERS = 2
DROPOUT    = 0.1
POS_WEIGHT = 4.03
BATCH_SIZE = 2048
MAX_EPOCHS = 80
PATIENCE   = 10
N_FOLDS    = 5

print(f'Static features: {len(FEAT_COLS)}, Seq features: {len(SEQ_COLS)}, Window: {WINDOW}, d_model: {D_MODEL}')
if DEVICE.type == 'cpu':
    print('WARNING: CPU-only. Upload to Kaggle T4 for 30-45 min training.')

## Feature Sets and Hyperparameters

Three distinct input types map to TFT's architectural roles:

| Set | Size | Role in TFT |
|-----|------|-------------|
| `FEAT_COLS` (static scalars) | 38 | Fed into Static VSN, conditions LSTM init |
| `SEQ_COLS` (temporal raw) | 6 | Fed into Temporal VSN per lap in the window |
| `CAT_COLS` (categoricals) | 4 | Entity embeddings → `entity_ctx` used as VSN context |

**Why raw columns in `SEQ_COLS`, not pre-engineered lag scalars?** The Temporal VSN learns which of the 6 raw features matter at each timestep across the full 15-lap window. Feeding `LapTime_lag1/2/3` as sequence features would duplicate information already present as static scalars in `FEAT_COLS` (where those lags also appear), creating confusing redundancy that the GRN gating would have to unlearn.

**Hyperparameter rationale:**
- `WINDOW=15`: Shorter than NB18's 20 because multi-head self-attention over LSTM outputs captures cross-timestep dependencies beyond what local LSTM hidden states alone carry. Reduces the per-sample tensor size for faster T4 training.
- `D_MODEL=64`: The Static VSN alone creates 38 × 64 = 2,432 parameters (per-feature projections). D_MODEL=64 balances expressiveness with the T4 compute budget (~30–45 min/fold). Doubling to 128 quadruples VSN parameter count.
- `lr=5e-4`: Slightly higher than the standard 1e-3 (which can destabilise VSN softmax weights early in training). CosineAnnealingLR decays to ~0 over T_max=50 epochs, letting the VSN gating settle smoothly.
- `POS_WEIGHT=4.03`: Positive class frequency = 19.9% of 439,140 train rows (~87,500 pit laps). Weight = (1 − 0.199) / 0.199 ≈ 4.03, correcting the 4:1 negative:positive imbalance.

In [ ]:
# nb19-04  Label encoders for entity embeddings (fit on combined train+test)
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

# vocab_size+1 reserves index 0 for padding
EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1,  8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1,  3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1,  2),
}
print('Embedding dims:', EMB_DIMS)

## Entity Embeddings as VSN Context

Entity embeddings serve **two roles** in NB19, unlike NB18 where they were concatenated only at the output head:

1. **Projected to `entity_ctx`** via `emb_proj(45 → d_model)` — this context vector is passed to *both* the Static VSN and the Temporal VSN's weight-selection GRN. The VSN learns which features matter **conditional on driver/race identity**: a driver known for aggressive tyre management may activate different VSN weights than a conservative one, even at the same TyreLife value.
2. **Incorporated into `static_repr`** via the merged GRN — race and driver identity directly influence the LSTM initial hidden and cell states (`h0`, `c0`), shaping the entire sequential dynamics from step 1.

This dual role means driver/race identity influences feature selection, sequential dynamics, and the final logit — not just a single concatenation point.

**Embedding dimensions** (proportional to log₂(cardinality) rounded to nearest practical size):

| Category | Unique values | Embedding dim |
|---|---|---|
| Driver | 887 | 32 |
| Race | 26 | 8 |
| Compound | 5 | 3 |
| Year | 4 | 2 |
| **Total** | — | **45** |

Label encoders are fit on **combined train + test** to prevent unseen-index errors at inference. Test has no unseen drivers or races vs train (confirmed from parquet). `vocab_size + 1` reserves index 0 for zero-padding: rows with fewer than `WINDOW` preceding laps are left-padded with zeros, so the embedding lookup at index 0 produces a learned "padding" vector.

In [ ]:
# nb19-05  Build 15-lap sequence windows
idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

# Deduplicate: SEQ_COLS and FEAT_COLS share some columns
all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + FEAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)

combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)
combined_df[SEQ_COLS] = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

N        = len(combined_df)
N_SF     = len(SEQ_COLS)
seq_vals = combined_df[SEQ_COLS].values.astype(np.float32)

all_windows = np.zeros((N, WINDOW, N_SF), dtype=np.float32)
all_masks   = np.zeros((N, WINDOW),       dtype=bool)

GROUP_COLS = ['Race', 'Year', 'Driver']
print(f'Building {WINDOW}-lap windows for {N:,} rows...')
t0 = time.time()

for _, grp_idx in tqdm(combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
                       total=combined_df[GROUP_COLS].drop_duplicates().shape[0]):
    idxs  = grp_idx.values
    n_grp = len(idxs)
    for i in range(n_grp):
        hist_len = min(i, WINDOW)
        if hist_len > 0:
            all_windows[idxs[i], WINDOW - hist_len:] = seq_vals[idxs[i - hist_len : i]]
            all_masks[idxs[i],   WINDOW - hist_len:] = True

print(f'Done in {time.time()-t0:.1f}s')
print(f'Rows with full history (>=15 prior laps): {all_masks.all(axis=1).sum():,}')
print(f'Rows with zero history: {(~all_masks.any(axis=1)).sum():,}')

In [ ]:
# nb19-06  Split windows back to train / test arrays
tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

train_windows   = all_windows[tr_mask]
train_seq_masks = all_masks[tr_mask]
test_windows    = all_windows[te_mask]
test_seq_masks  = all_masks[te_mask]

train_static_raw = combined_df.loc[tr_mask, FEAT_COLS].values.astype(np.float32)
test_static_raw  = combined_df.loc[te_mask, FEAT_COLS].values.astype(np.float32)
train_targets    = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
train_ids        = combined_df.loc[tr_mask, 'id'].values
test_ids         = combined_df.loc[te_mask, 'id'].values

fold_lookup = folds_df.set_index('id')['fold']
train_folds = fold_lookup.loc[train_ids].values

train_cat = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
test_cat  = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}

del all_windows, all_masks, combined_df

print(f'Train windows: {train_windows.shape},  static: {train_static_raw.shape}')
print(f'Test  windows: {test_windows.shape},  static: {test_static_raw.shape}')
print(f'Fold counts: {pd.Series(train_folds).value_counts().sort_index().to_dict()}')

## Model Architecture: GRN + VSN + TFT Encoder

The entire TFT Encoder is assembled from two reusable building blocks:

### Gated Residual Network (GRN)

The core building block used throughout — takes input `a` and optional context `c`:

```
skip  = Linear(d_in → d_model)(a)         [identity if d_in == d_model]
x     = LayerNorm(a)
x     = cat([x, c], dim=-1)               [if context provided; c is broadcast over time dim]
x     = Dropout(ELU(fc1(x)))
v, g  = fc2(x).chunk(2, dim=-1)           [gating: value stream + gate stream]
out   = LayerNorm(skip + v × sigmoid(g))  [residual + Gated Linear Unit]
```

The `sigmoid(g)` gate suppresses noise — if the network is uncertain, it can zero out `v` while preserving the residual `skip`. Works identically on static `(B, d)` and temporal `(B, T, d)` tensors; context `c=(B, d_context)` is unsqueezed and expanded over the time dimension when needed.

### Variable Selection Network (VSN)

```
For each of n features i:
  proj_i: Linear(1 → d_model)   →  feature_repr_i  (..., d_model)
Stack all n projections          →  stacked  (..., n, d_model)
Flatten                          →  flat     (..., n × d_model)
GRN(flat, context) → Linear(d_model → n) → softmax  →  weights  (..., n)
Weighted sum of stacked:         →  (..., d_model)
LayerNorm                        →  output
```

The softmax weights sum to 1 — a sparse-ish gating over the `n` feature projections. The optional `context` (entity_ctx for static VSN; static_repr for temporal VSN) conditions the weight distribution on driver/race/compound identity.

### Forward Pass Annotations

| Step | What happens | Why |
|------|-------------|-----|
| Entity embeddings → `entity_ctx` | 45-dim cat embedding → Linear(45→64) + ReLU | Compact driver/race context for VSN conditioning |
| Static VSN | 38 scalar features × per-feature Linear(1→64) → softmax gating | Learn which scalar features matter for this race/driver |
| Static merge GRN | cat[static_vsn, entity_ctx] → GRN(128→64) | Fuse feature-selected representation with identity context |
| LSTM init | Linear(64→2×64) → h0, c0 | Static context shapes LSTM hidden dynamics from step 1, not just the final readout |
| Temporal VSN | 6 raw features × 15 laps → per-feature Linear(1→64) → softmax gating (ctx=static_repr) | Which of the 6 sequence features matters at each lap, conditioned on race context |
| LSTM | Processes VSN output; pack_padded ignores zero-padded steps | Sequential dynamics; padding mask prevents ghost gradient from padded laps |
| Multi-head self-attention | 4 heads over LSTM outputs; key_padding_mask=~mask | Capture long-range patterns (lap 5 behaviour attends to lap 35) beyond LSTM's local horizon |
| Attn GRN residual | GRN(LayerNorm(lstm_out + attn_out)) | Stabilises the attention residual; can down-weight attention when unhelpful |
| Masked mean pool | (attn_out × valid_mask).sum(1) / valid_count | Gives each valid timestep equal weight; more robust than last-hidden-state for variable-length histories |
| Output head | GRN(pooled) → Linear(64→1) | Final logit |

**Gradient clipping (`max_norm=1.0`):** VSN softmax weights are sensitive to large gradient norms during early training when the gating is still learning to differentiate features. Clipping prevents the gating from collapsing to near-one-hot too quickly.

In [ ]:
# nb19-07  GRN + VSN + TFT Encoder model

class GRN(nn.Module):
    """Gated Residual Network: LayerNorm -> ELU -> GLU -> residual + LayerNorm."""
    def __init__(self, d_in, d_model, d_context=0, dropout=0.05):
        super().__init__()
        self.skip   = nn.Linear(d_in, d_model) if d_in != d_model else nn.Identity()
        self.ln_in  = nn.LayerNorm(d_in)
        self.fc1    = nn.Linear(d_in + d_context, d_model)
        self.fc2    = nn.Linear(d_model, d_model * 2)
        self.ln_out = nn.LayerNorm(d_model)
        self.drop   = nn.Dropout(dropout)

    def forward(self, a, c=None):
        # a: (..., d_in);  c: (B, d_context) optionally broadcast over time dim
        skip = self.skip(a)
        x    = self.ln_in(a)
        if c is not None:
            if c.dim() < x.dim():
                c = c.unsqueeze(1).expand(*x.shape[:-1], c.shape[-1])
            x = torch.cat([x, c], dim=-1)
        x    = self.drop(F.elu(self.fc1(x)))
        v, g = self.fc2(x).chunk(2, dim=-1)
        return self.ln_out(skip + v * torch.sigmoid(g))


class VSN(nn.Module):
    """Variable Selection Network: learns softmax weights over per-feature projections."""
    def __init__(self, n_feats, d_model, d_context=0, dropout=0.05):
        super().__init__()
        self.projs  = nn.ModuleList([nn.Linear(1, d_model) for _ in range(n_feats)])
        self.w_grn  = GRN(n_feats * d_model, d_model, d_context=d_context, dropout=dropout)
        self.w_fc   = nn.Linear(d_model, n_feats)
        self.ln_out = nn.LayerNorm(d_model)
        self.n, self.d = n_feats, d_model

    def forward(self, x, context=None):
        # x: (..., n_feats) — works for both (B, n) static and (B, T, n) temporal
        parts   = [p(x[..., i:i+1]) for i, p in enumerate(self.projs)]
        stacked = torch.stack(parts, dim=-2)                           # (..., n, d)
        flat    = stacked.view(*stacked.shape[:-2], self.n * self.d)   # (..., n*d)
        w       = F.softmax(self.w_fc(self.w_grn(flat, context)), dim=-1)  # (..., n)
        return self.ln_out((stacked * w.unsqueeze(-1)).sum(dim=-2))    # (..., d)


class TFTEncoder(nn.Module):
    """Simplified TFT Encoder: VSN feature selection + static-init LSTM + temporal self-attention."""
    def __init__(self, n_static=38, n_seq=6, d_model=64, num_heads=4,
                 num_lstm_layers=2, dropout=0.1, emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS
        emb_total = sum(v[1] for v in emb_dims.values())  # 45

        # Entity embeddings
        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        self.emb_proj     = nn.Linear(emb_total, d_model)

        # Static path: VSN + merge with entity context
        self.static_vsn   = VSN(n_static, d_model, d_context=d_model, dropout=dropout)
        self.static_merge = GRN(d_model * 2, d_model, dropout=dropout)

        # Init LSTM states from static representation
        self.h0_net = nn.Linear(d_model, num_lstm_layers * d_model)
        self.c0_net = nn.Linear(d_model, num_lstm_layers * d_model)

        # Temporal path: VSN (context = static repr)
        self.temporal_vsn = VSN(n_seq, d_model, d_context=d_model, dropout=dropout)

        # LSTM encoder
        self.lstm = nn.LSTM(d_model, d_model, num_lstm_layers, batch_first=True,
                            dropout=dropout if num_lstm_layers > 1 else 0.0)

        # Temporal self-attention + gated residual
        self.attn     = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.attn_ln  = nn.LayerNorm(d_model)
        self.attn_grn = GRN(d_model, d_model, dropout=dropout)

        # Output
        self.out_grn = GRN(d_model, d_model, dropout=dropout)
        self.head    = nn.Linear(d_model, 1)

        self.d = d_model
        self.L = num_lstm_layers

    def forward(self, window, mask, static, driver, race, compound, year):
        B, T, _ = window.shape

        # Entity context
        emb = torch.cat([self.driver_emb(driver), self.race_emb(race),
                         self.compound_emb(compound), self.year_emb(year)], dim=1)
        entity_ctx = F.relu(self.emb_proj(emb))                          # (B, d)

        # Static VSN + merge
        static_vsn  = self.static_vsn(static, context=entity_ctx)        # (B, d)
        static_repr = self.static_merge(
            torch.cat([static_vsn, entity_ctx], dim=-1))                  # (B, d)

        # LSTM initial states from static
        h0 = self.h0_net(static_repr).view(B, self.L, self.d).permute(1, 0, 2).contiguous()
        c0 = self.c0_net(static_repr).view(B, self.L, self.d).permute(1, 0, 2).contiguous()

        # Temporal VSN
        tvn = self.temporal_vsn(window, context=static_repr)             # (B, T, d)

        # LSTM over valid sequence steps
        seq_len = mask.sum(1).long().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            tvn, seq_len.cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed, (h0, c0))
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            lstm_out, batch_first=True, total_length=T)                  # (B, T, d)

        # Safe attention mask: rows with zero valid laps have mask all-False, so ~mask is
        # all-True — MultiheadAttention softmax over all -inf = NaN. Fall back to unmasked
        # attention for those rows (they are all-padding anyway; any attended position is fine).
        attn_key_mask = ~mask                                             # True = ignore
        no_valid = attn_key_mask.all(dim=1)                              # (B,)
        if no_valid.any():
            attn_key_mask = attn_key_mask.clone()
            attn_key_mask[no_valid] = False                              # attend everywhere

        attn_out, _ = self.attn(lstm_out, lstm_out, lstm_out,
                                key_padding_mask=attn_key_mask)
        attn_out    = self.attn_grn(self.attn_ln(lstm_out + attn_out))   # (B, T, d)

        # Masked mean pool over valid positions
        valid  = mask.float().unsqueeze(-1)                              # (B, T, 1)
        pooled = (attn_out * valid).sum(1) / valid.sum(1).clamp(min=1)  # (B, d)

        return self.head(self.out_grn(pooled)).squeeze(1)               # (B,)


_m = TFTEncoder(n_static=38, n_seq=len(SEQ_COLS), d_model=D_MODEL,
                num_heads=NUM_HEADS, num_lstm_layers=NUM_LAYERS, dropout=DROPOUT)
n_params = sum(p.numel() for p in _m.parameters())
print(f'TFT Encoder params: {n_params:,}')
del _m

In [ ]:
# nb19-08  Dataset + DataLoader
class F1SeqDataset(Dataset):
    def __init__(self, windows, masks, static, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.static   = torch.tensor(static,  dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self):  return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], static=self.static[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, static, cat_idxs, targets=None, shuffle=False):
    ds = F1SeqDataset(windows, masks, static, cat_idxs, targets)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
# nb19-09  Training utilities
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_logits = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            stat = batch['static'].to(DEVICE)
            drv  = batch['driver'].to(DEVICE)
            rc   = batch['race'].to(DEVICE)
            cmp  = batch['compound'].to(DEVICE)
            yr   = batch['year'].to(DEVICE)

            logits = model(win, mask, stat, drv, rc, cmp, yr)

            if is_train:
                tgt  = batch['target'].to(DEVICE)
                loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_logits.append(logits.detach().cpu().float().numpy())

    probs = torch.sigmoid(torch.tensor(np.concatenate(all_logits))).numpy()
    return probs, total_loss / max(len(loader.dataset), 1)

## 5-Fold GroupKFold Cross-Validation

Same validation protocol as all Phase 3 notebooks:

1. **GroupKFold(5) by Race + Year (42 groups):** Whole races are assigned to a single fold. No lap from Race X appears in both train and validation — simulates predicting pit stops in races the model has never seen during training.
2. **Static feature scaling:** `StandardScaler` fit on the training fold only; applied to validation and test. Sequence features (`SEQ_COLS`) were scaled globally in cell nb19-05 (fit on all train rows) because the window construction requires a single consistent scale across all laps before splitting.
3. **Fresh model per fold:** New `TFTEncoder` weights initialised from scratch. No weight sharing across folds — each fold model is an independent estimate of generalisation performance.
4. **AdamW + CosineAnnealingLR:** Weight decay regularises entity embedding tables and VSN projection weights (which would otherwise overfit to frequent driver/race combinations). Cosine schedule decays lr from `5e-4` to ~0 over `T_max=50` epochs, letting VSN gating weights settle gradually.
5. **Early stopping (patience=10):** Validation AUC monitored after every epoch. Training stops if no improvement for 10 consecutive epochs. Best model state is checkpointed and reloaded for OOF and test inference.
6. **Fold model saved:** `models/tft_fold{fold}.pt` stores the best-AUC checkpoint. Test predictions are averaged across all 5 fold models to reduce variance.

In [ ]:
# nb19-10  5-fold CV training
oof_preds        = np.zeros(len(train_ids))
test_preds_folds = []
fold_aucs        = []

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE)
)

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'Fold {fold}')
    print('='*60)

    tr_idx = np.where(train_folds != fold)[0]
    va_idx = np.where(train_folds == fold)[0]

    scaler  = StandardScaler()
    tr_stat = scaler.fit_transform(train_static_raw[tr_idx])
    va_stat = scaler.transform(train_static_raw[va_idx])
    te_stat = scaler.transform(test_static_raw)

    tr_cat = {c: train_cat[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: train_cat[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(train_windows[tr_idx], train_seq_masks[tr_idx],
                            tr_stat, tr_cat, targets=train_targets[tr_idx], shuffle=True)
    va_loader = make_loader(train_windows[va_idx], train_seq_masks[va_idx], va_stat, va_cat)
    te_loader = make_loader(test_windows, test_seq_masks, te_stat, test_cat)

    model     = TFTEncoder(n_static=38, n_seq=len(SEQ_COLS), d_model=D_MODEL,
                           num_heads=NUM_HEADS, num_lstm_layers=NUM_LAYERS,
                           dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(MAX_EPOCHS):
        _, tr_loss   = run_epoch(model, tr_loader, criterion, optimizer)
        scheduler.step()
        val_probs, _ = run_epoch(model, va_loader, criterion)
        val_auc      = roc_auc_score(train_targets[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc, best_epoch, patience_ctr = val_auc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0 or patience_ctr >= PATIENCE:
            print(f'  ep {epoch+1:3d}  loss={tr_loss:.4f}  val={val_auc:.4f}  '
                  f'best={best_auc:.4f} (ep{best_epoch+1})')

        if patience_ctr >= PATIENCE:
            print('  Early stop')
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    oof_probs, _ = run_epoch(model, va_loader, criterion)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(train_targets[va_idx], oof_probs)
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold} AUC: {fold_auc:.4f}')

    te_probs, _ = run_epoch(model, te_loader, criterion)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'tft_fold{fold}.pt')
    print(f'  Saved tft_fold{fold}.pt  (best ep {best_epoch+1})')

oof_auc = roc_auc_score(train_targets, oof_preds)
print(f'\nOOF AUC : {oof_auc:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')

In [ ]:
# nb19-11  Correlation analysis + gate check + ensemble previews
id_idx   = pd.Index(train_ids)
mlp_vals  = mlp_oof.set_index('id').loc[train_ids, 'mlp_oof'].values
lstm_vals = lstm_oof.set_index('id').loc[train_ids, 'lstm_oof'].values
lgbm_vals = lgbm_oof.set_index('id').loc[train_ids, 'lgbm_tuned_oof'].values

rho_mlp,  _ = spearmanr(oof_preds, mlp_vals)
rho_lstm, _ = spearmanr(oof_preds, lstm_vals)
rho_lgbm, _ = spearmanr(oof_preds, lgbm_vals)
rho_ml,   _ = spearmanr(mlp_vals, lstm_vals)

mlp_auc  = roc_auc_score(train_targets, mlp_vals)
lstm_auc = roc_auc_score(train_targets, lstm_vals)
lgbm_auc = roc_auc_score(train_targets, lgbm_vals)

print(f'TFT  solo OOF AUC : {oof_auc:.4f}')
print(f'MLP  solo OOF AUC : {mlp_auc:.4f}')
print(f'LSTM solo OOF AUC : {lstm_auc:.4f}')
print(f'LGBM solo OOF AUC : {lgbm_auc:.4f}')
print()
print('Spearman rho matrix:')
print(f'  TFT  x MLP : {rho_mlp:.4f}')
print(f'  TFT  x LSTM: {rho_lstm:.4f}')
print(f'  TFT  x LGBM: {rho_lgbm:.4f}')
print(f'  MLP  x LSTM: {rho_ml:.4f}')
print()
print(f'Gate: solo AUC >= 0.905  -> {"PASS" if oof_auc >= 0.905 else "FAIL"}')
print(f'Gate: rho  <  0.90 vs MLP -> {"PASS" if rho_mlp < 0.90 else "FAIL"}')

# Rank-average ensemble previews
def rank_norm(a): return rankdata(a) / len(a)

r_tft  = rank_norm(oof_preds)
r_mlp  = rank_norm(mlp_vals)
r_lstm = rank_norm(lstm_vals)
r_lgbm = rank_norm(lgbm_vals)

auc_tft_mlp       = roc_auc_score(train_targets, (r_tft + r_mlp) / 2)
auc_tft_mlp_lgbm  = roc_auc_score(train_targets, (r_tft + r_mlp + r_lgbm) / 3)
auc_tft_mlp_lstm  = roc_auc_score(train_targets, (r_tft + r_mlp + r_lstm) / 3)
auc_all4          = roc_auc_score(train_targets, (r_tft + r_mlp + r_lstm + r_lgbm) / 4)

print(f'\nRank avg TFT+MLP:            {auc_tft_mlp:.4f}  (delta vs MLP: {auc_tft_mlp - mlp_auc:+.4f})')
print(f'Rank avg TFT+MLP+LGBM:       {auc_tft_mlp_lgbm:.4f}')
print(f'Rank avg TFT+MLP+LSTM:       {auc_tft_mlp_lstm:.4f}')
print(f'Rank avg TFT+MLP+LSTM+LGBM:  {auc_all4:.4f}')

## Correlation Analysis and Phase 3 Gate

**Spearman ρ** measures rank correlation between OOF prediction vectors. Lower ρ means the two models make errors on different rows — their rank averages help each other more.

### Gate Thresholds

| Gate | Threshold | Reasoning |
|------|-----------|-----------|
| Solo AUC | ≥ 0.905 | Raised bar: best solo is LSTM at 0.9122. A weaker model added to rank average dilutes stronger models' rank ordering (confirmed: ET 0.8788 dragged LGBM+CB from 0.9032 to 0.8985). |
| ρ vs MLP | < 0.90 | LSTM failed this gate at ρ=0.9126 — no CV gain from LSTM+MLP over MLP+LGBM. High ρ means shared error structure that rank averaging cannot resolve. |

### TFT Diversity Hypothesis

The VSN's dynamic feature routing is architecturally distinct from MLP's fixed global weights. The temporal self-attention can attend to distant laps (e.g., lap 5 stint choice influencing lap 35 prediction), while the MLP sees only fixed engineered lag windows. Expected ρ vs MLP: ~0.78–0.83.

**Root cause of LSTM's ρ failure:** LSTM window features [LapTime, TyreLife, Cum_Deg, LapTime_Delta, Position] contain the same tyre trajectory information already encoded by MLP's lag scalars. TFT's VSN may route differently — if VSN selects `LapTime_Delta` + `Position` over raw `LapTime`, the error pattern diverges from MLP's lag-dominant representation.

### Gate Outcomes

| Result | Action |
|--------|--------|
| PASS both gates | Include `tft_oof` in NB20 rank average |
| FAIL AUC only (< 0.905) | Exclude from NB20; save outputs for reference |
| FAIL ρ only (≥ 0.90 vs MLP) | Save outputs; NB20 checks ρ(TFT, LGBM) separately — TFT may still contribute diversity through the LGBM path even if correlated with MLP |

The ensemble preview deltas show incremental value of adding TFT. A delta > +0.001 over the existing best ensemble is meaningful; sub-+0.001 gains do not reliably transfer to LB (confirmed v002 vs v003: +0.0008 CV = +0.00005 LB).

In [ ]:
# nb19-12  Average test predictions across folds
test_preds_mean = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
print(f'Test preds: shape={test_preds_mean.shape}, mean={test_preds_mean.mean():.4f}, '
      f'std={test_preds_mean.std():.4f}')

In [ ]:
# nb19-13  Save outputs
oof_df = pd.DataFrame({
    'id':         train_ids,
    'fold':       train_folds,
    'PitNextLap': train_targets.astype(int),
    'tft_oof':    oof_preds,
})
assert len(oof_df) == 439140, f'Expected 439140 rows, got {len(oof_df)}'
assert oof_df['id'].nunique() == 439140, 'Duplicate ids in OOF'
oof_df.to_parquet(processed_dir / 'oof_predictions_nb19.parquet', index=False)
print(f'Saved oof_predictions_nb19.parquet  ({len(oof_df):,} rows)')

test_out = pd.DataFrame({'id': test_ids, 'tft_pred': test_preds_mean})
test_out.to_parquet(processed_dir / 'test_predictions_nb19.parquet', index=False)
print(f'Saved test_predictions_nb19.parquet ({len(test_out):,} rows)')

summary = {
    'oof_auc':           oof_auc,
    'fold_aucs':         fold_aucs,
    'fold_std':          float(np.std(fold_aucs)),
    'rho_vs_mlp':        float(rho_mlp),
    'rho_vs_lstm':       float(rho_lstm),
    'rho_vs_lgbm':       float(rho_lgbm),
    'auc_tft_mlp':       float(auc_tft_mlp),
    'auc_all4':          float(auc_all4),
    'model':             'TFT-Encoder-NB19',
    'architecture':      f'VSN({len(FEAT_COLS)}+{len(SEQ_COLS)})+StaticInit-LSTM({D_MODEL}x{NUM_LAYERS})+Attn({NUM_HEADS}h)',
    'window_size':       WINDOW,
    'd_model':           D_MODEL,
    'gate_pass':         bool(oof_auc >= 0.905 and rho_mlp < 0.90),
}
with open(models_dir / 'nb19_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print(f'\n=== NB19 Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

## Summary

### Artifacts Saved

| File | Description |
|------|-------------|
| `data/processed/oof_predictions_nb19.parquet` | 439,140 rows: `id`, `fold`, `PitNextLap`, `tft_oof` |
| `data/processed/test_predictions_nb19.parquet` | Test rows: `id`, `tft_pred` (avg of 5 fold models) |
| `models/tft_fold{0-4}.pt` | Best-epoch checkpoint per fold |
| `models/nb19_summary.pkl` | OOF AUC, fold AUCs, ρ values, gate_pass flag |

### NB20 Inputs (Super-Ensemble)

NB20 assembles rank averages across all Phase 3 outputs:
- `oof_predictions_nb12.parquet` → `lgbm_tuned_oof` (LGBM anchor, CV 0.9024)
- `oof_predictions_nb15.parquet` → `mlp_oof` (MLP, CV 0.9102)
- `oof_predictions_nb18.parquet` → `lstm_oof` (LSTM, CV 0.9122; ρ gate FAIL vs MLP, but may contribute via ρ(LSTM, LGBM))
- `oof_predictions_nb19.parquet` → `tft_oof` **(this notebook)**
- `oof_predictions_nb21.parquet` → `hybrid_oof` (NB21 Hybrid GRU+FC — not yet built)

NB20 prints the full Spearman ρ matrix across all models before assembling any ensemble.

### CV→LB Projection

Using the confirmed +0.0166 MLP-ensemble boost (v004, any ensemble containing MLP or sequence models):

| Scenario | Expected CV | Proj LB |
|----------|-------------|---------|
| TFT passes gate → 4-model LGBM+MLP+LSTM+TFT | ~0.930–0.938 | ~0.946–0.955 |
| TFT fails gate → 3-model LGBM+MLP+LSTM | ~0.924 | ~0.940 |

### Next: NB21 — Multi-Input Hybrid GRU+FC

NB21 uses **two completely separate processing branches** that never share inputs — the key inductive bias distinguishing it from NB18 (all features in sequence) and NB19 (unified VSN routing):

- **GRU branch:** 10-lap window of pure temporal features [LapTime, TyreLife, Cum_Deg, LapTime_Delta, Position] → hidden=128. The GRU never sees strategic context.
- **FC branch:** 13 strategic scalars [Stint, RaceProgress, laps_remaining, compound_ordinal, is_wet, prime_pit_window, laps_to_driver_end, abs_position_change, pos_change_rolling_std_3, PitStop_lag1, interaction features] → FC 64 units. The FC branch never processes a sequence.
- **Merge:** cat[gru(128), fc(64), embeddings(45)] → head. Entity embeddings are shared, not duplicated.

This split mirrors how F1 strategists actually think: tyre trajectory and race strategic context are separate sub-problems that only need to be reconciled at the final decision point.